# 09 — Model Evaluation

## Purpose

This notebook provides the formal evaluation of the selected Random Forest
model for poultry disease outbreak-count forecasting.

The evaluation focuses on chronological, out-of-sample performance rather than
training-set performance.

The selected model was identified in Notebook 08 using rolling-origin temporal
validation. This notebook evaluates its predictive performance, error behaviour,
temporal stability, and deployment implications.

### Evaluation objectives

1. Load the final saved model and associated metadata.
2. Reconstruct the evaluation data and chronological validation folds.
3. Generate out-of-sample predictions for each validation period.
4. Calculate MAE, RMSE, and Median Absolute Error.
5. Compare performance across chronological periods.
6. Analyse prediction errors and large deviations.
7. Assess temporal stability and potential model drift.
8. Document the model's strengths, limitations, and deployment implications.

> Training-set metrics are not treated as the primary measure of model
> performance because they are optimistic. The main performance evidence
> comes from chronological out-of-sample validation.

In [1]:
# ==============================================================
# MODEL EVALUATION — IMPORTS
# ==============================================================
# This cell imports the libraries required for evaluating the
# selected Random Forest model.
# ==============================================================

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
)

print("=" * 70)
print("MODEL EVALUATION — INITIALISATION")
print("=" * 70)

print("\nRequired libraries imported successfully.")

MODEL EVALUATION — INITIALISATION

Required libraries imported successfully.


In [4]:
# ==============================================================
# INSPECT LOADED MODEL METADATA
# ==============================================================
# The metadata file was created in Notebook 08.
# Rather than assuming specific JSON key names, we inspect the
# available metadata fields first.
# ==============================================================

print("\nAvailable metadata fields:")

for key, value in evaluation_metadata.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 70)
print("ARTEFACT LOADING PASSED")
print("=" * 70)


Available metadata fields:
  model_name: Random Forest
  model_type: RandomForestRegressor
  model_version: 1.0.0
  target_column: target_outbreak_count
  n_training_records: 445
  n_features: 27
  features: ['year', 'semester_order', 'is_first_semester', 'is_second_semester', 'semester_sin', 'semester_cos', 'years_since_2019', 'year_sin', 'year_cos', 'has_adm1', 'is_country_level', 'disease_code', 'previous_semester_outbreak_count', 'two_semesters_prior_outbreak_count', 'outbreak_history_mean_3', 'outbreak_history_max_3', 'climate_available', 'temperature_missing', 'dewpoint_missing', 'precipitation_missing', 'wind_missing', 'climate_variable_count', 'climate_complete', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms']
  training_period: {'start_period': 4041, 'end_period': 4052}
  validation: {'method': 'rolling_origin_temporal_validation', 'folds': 8, 'mean_fold_mae': 6.533, 'std_fold_mae': 1.661, 'mean_fold_rmse': 11.603

In [5]:
# Load the saved Random Forest pipeline.
evaluation_model = joblib.load(model_path)

# Load model metadata.
with open(metadata_path, "r", encoding="utf-8") as file:
    evaluation_metadata = json.load(file)

# Load the production feature schema.
with open(schema_path, "r", encoding="utf-8") as file:
    production_schema = json.load(file)

print("\nModel artefacts loaded successfully.")

# ==============================================================
# INSPECT LOADED MODEL METADATA
# ==============================================================
# The metadata file was created in Notebook 08.
# Rather than assuming specific JSON key names, we inspect the
# available metadata fields first.
# ==============================================================

print("\nAvailable metadata fields:")

for key, value in evaluation_metadata.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 70)
print("ARTEFACT LOADING PASSED")
print("=" * 70)


Model artefacts loaded successfully.

Available metadata fields:
  model_name: Random Forest
  model_type: RandomForestRegressor
  model_version: 1.0.0
  target_column: target_outbreak_count
  n_training_records: 445
  n_features: 27
  features: ['year', 'semester_order', 'is_first_semester', 'is_second_semester', 'semester_sin', 'semester_cos', 'years_since_2019', 'year_sin', 'year_cos', 'has_adm1', 'is_country_level', 'disease_code', 'previous_semester_outbreak_count', 'two_semesters_prior_outbreak_count', 'outbreak_history_mean_3', 'outbreak_history_max_3', 'climate_available', 'temperature_missing', 'dewpoint_missing', 'precipitation_missing', 'wind_missing', 'climate_variable_count', 'climate_complete', 'temperature_c', 'dewpoint_c', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms']
  training_period: {'start_period': 4041, 'end_period': 4052}
  validation: {'method': 'rolling_origin_temporal_validation', 'folds': 8, 'mean_fold_mae': 6.533, 'std_fold

## 2. Evaluation Dataset

The final Random Forest model was trained on 445 eligible historical
observations using 27 predictor variables.

For formal evaluation, we will reconstruct the modelling dataset used during
the temporal validation process.

The evaluation will preserve the chronological structure of the data and will
not use random train/test splitting.

This ensures that future observations are never used to evaluate predictions
for earlier periods.

In [6]:
# ==============================================================
# LOAD MODELLING DATA FOR EVALUATION
# ==============================================================
# This cell loads the project's processed modelling dataset.
#
# We first inspect the available project files so that we use
# the correct existing data source rather than creating a new
# dataset or guessing a filename.
# ==============================================================

# Display the project root being used.
print(f"Project root: {project_root}")

# Define the directories where processed data is expected.
data_directory = project_root / "data"
processed_directory = data_directory / "processed"

print("\nChecking data directories:")
print(f"Data directory exists:       {data_directory.exists()}")
print(f"Processed directory exists:  {processed_directory.exists()}")

# List files in the processed-data directory.
if processed_directory.exists():
    processed_files = sorted(processed_directory.iterdir())

    print("\nFiles in data/processed:")
    
    if processed_files:
        for file_path in processed_files:
            print(f"  - {file_path.name}")
    else:
        print("  No files found.")
else:
    raise FileNotFoundError(
        f"Processed data directory not found: {processed_directory}"
    )

Project root: c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

Checking data directories:
Data directory exists:       True
Processed directory exists:  True

Files in data/processed:
  - README.md
  - wahis_administrative_boundary_lookup.csv
  - wahis_analytical_table.csv
  - wahis_climate_integrated.csv
  - wahis_modelling_features.csv
  - wahis_quantitative_cleaned.csv
